# Postcode → LSOA Preprocessing

Builds a lightweight **postcode → LSOA** lookup table from the raw ONS
postcode directory.

- **Input:** `../data/bronze/postcode/ONSPD_FEB_2024_UK.csv` (~1.3 GB, 51 cols)
- **Output:** `../data/silver/postcode_lsoa.csv` (2 cols)

Only `pcds` (postcode, single-space format) and `lsoa11` (2011 Census LSOA,
matches the 2017-2021 ADI data) are extracted. Duplicates are removed on
postcode.

In [ ]:
import os
import pandas as pd

## 1. Load (only the columns we need)

The source file is ~1.3 GB / 2.7M rows. We read only the two required
columns via `usecols` (as strings) instead of loading all 51 columns, to
keep memory usage low.

In [ ]:
RAW = "../data/bronze/postcode/ONSPD_FEB_2024_UK.csv"


def standardise_columns(df):
    """Standardise column names: strip, lowercase, spaces -> underscores,
    then rename the raw ONS identifiers to the project's terminology
    (pcds -> postcode, lsoa11 -> lsoa_code).
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    df = df.rename(columns={"pcds": "postcode", "lsoa11": "lsoa_code"})
    return df


# Read only the postcode + 2011 LSOA columns, as strings
df = pd.read_csv(RAW, usecols=["pcds", "lsoa11"], dtype=str)
df = standardise_columns(df)
print(df.shape)
df.head()

## 2. Clean & deduplicate

In [ ]:
def clean_postcode_lsoa(df):
    """Clean the postcode lookup: trim whitespace, drop rows with a
    missing/blank postcode or lsoa_code (terminated postcodes have no
    LSOA), and remove duplicate postcodes (keep first occurrence).
    """
    df = df.copy()

    # Trim surrounding whitespace on both fields
    df["postcode"] = df["postcode"].str.strip()
    df["lsoa_code"] = df["lsoa_code"].str.strip()

    # Drop nulls and blank strings in either column
    df = df.dropna(subset=["postcode", "lsoa_code"])
    df = df[(df["postcode"] != "") & (df["lsoa_code"] != "")]

    # Remove duplicate postcodes
    df = df.drop_duplicates(subset="postcode", keep="first")

    return df.reset_index(drop=True)


df = clean_postcode_lsoa(df)
print(df.shape)
df.head()

## 3. Validate

In [ ]:
print("shape:", df.shape)
print("duplicate postcodes:", df["postcode"].duplicated().sum())
display(df.isnull().sum())
df.head()

## 4. Export

In [ ]:
os.makedirs("../data/silver", exist_ok=True)
df.to_csv("../data/silver/postcode_lsoa.csv", index=False)
print("Wrote ../data/silver/postcode_lsoa.csv", df.shape)